# Fresh Baseline - Optimized YOLOv11n Baseline

## Goal
Train a fresh YOLOv11n from pretrained weights with an optimized training recipe.
Target: 78-80% mAP@0.5 on NEU-DET (baseline: 75.8%).

## Key Recipe Changes vs Previous Baselines
| Parameter | Old | New (4A) | Rationale |
|-----------|-----|----------|----------|
| mosaic | ~1.0 | 0.0 | DISABLED - was sabotaging fine defect features |
| mixup | 0.0 | 0.15 | Light mixup for regularization |
| copy_paste | 0.0 | 0.1 | Copy-paste augmentation |
| optimizer | auto (SGD) | AdamW | Better for fine-grained features |
| lr0 | 0.01 | 0.001 | Lower LR for AdamW |
| epochs | 300 | 600 | More time to converge |
| patience | 75 | 150 | More exploration room |
| imgsz | 800 | 800 | Kept - good for thin defects |
| cos_lr | True | True | Kept - better convergence |

## Rules
- NO custom modules - vanilla Ultralytics only
- YOLO('yolo11n.pt') - pretrained, NOT from YAML
- Every cell runs independently - no hidden state
- All paths use raw strings (Windows safe)

In [ ]:
# Cell 2: Environment Setup and Verification
import torch
import ultralytics
from pathlib import Path
import json
import os
import sys
import random
import numpy as np

# Seed for reproducibility
SEED = 42
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
np.random.seed(SEED)
random.seed(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

# Environment info
print(f"Python version:    {sys.version.split()[0]}")
print(f"PyTorch version:   {torch.__version__}")
print(f"Ultralytics ver:   {ultralytics.__version__}")
print(f"CUDA available:    {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU:               {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory:        {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    print("WARNING: No GPU detected - training will be extremely slow!")

# Paths
ROOT = Path(r"D:/DigiSteel-Yolo/DigiSteel-YOLO")
DATA_YAML = ROOT / "configs" / "data" / "neu_det.yaml"
RUNS_DIR = ROOT / "runs" / "detect"
RUN_NAME = "fresh_baseline"
BEST_PT = RUNS_DIR / RUN_NAME / "weights" / "best.pt"
RESULTS_CSV = RUNS_DIR / RUN_NAME / "results.csv"
EVALS_DIR = ROOT / "evals"
METRICS_JSON = EVALS_DIR / "fresh_baseline_results.json"
TRAIN_STATUS_FILE = EVALS_DIR / "fresh_baseline_train_status.json"

# Verify critical files exist
print(f"\n{'='*60}")
print("PATH VERIFICATION")
print(f"{'='*60}")

errors = []
if not DATA_YAML.exists():
    errors.append(f"Data YAML not found: {DATA_YAML}")
else:
    print(f"  Data YAML: {DATA_YAML}")

if not errors:
    import yaml
    with open(DATA_YAML, 'r') as f:
        data_cfg = yaml.safe_load(f)
    base_path = Path(data_cfg.get('path', ''))
    if not base_path.is_absolute():
        base_path = DATA_YAML.parent / base_path
    for split in ['train', 'val', 'test']:
        split_rel = data_cfg.get(split, '')
        if split_rel:
            split_path = base_path / split_rel
            if split_path.exists():
                count = len(list(split_path.glob('*.jpg'))) + len(list(split_path.glob('*.png')))
                print(f"  {split}: {split_path} ({count} images)")
            else:
                errors.append(f"{split} path not found: {split_path}")

EVALS_DIR.mkdir(parents=True, exist_ok=True)
RUNS_DIR.mkdir(parents=True, exist_ok=True)

if errors:
    print(f"\nERRORS - fix before proceeding:")
    for e in errors:
        print(f"  {e}")
    raise FileNotFoundError("Critical paths missing - see above.")

print(f"\nAll checks passed. Ready to train.")

---
## Cell 3: Load Pretrained YOLOv11n

Loading YOLOv11n from pretrained weights.
Ultralytics auto-downloads this file from the Ultralytics Hub if not found locally.

NOT loading from YAML - that would create a random-initialized model.

In [ ]:
# Cell 3: Build YOLOv11n from Architecture Definition + Load Pretrained Weights
from ultralytics import YOLO

# Step 1: Define YOLOv11n architecture from standard YAML
model = YOLO('yolo11.yaml')

print(f"{'='*60}")
print(f"ARCHITECTURE DEFINITION")
print(f"{'='*60}")
print(f"Architecture:  YOLOv11n (from YAML definition)")
print(f"Model type:    {type(model.model).__name__}")
print(f"Task:          {model.task}")
print(f"Parameters:    {sum(p.numel() for p in model.model.parameters()):,}")
print(f"Layers:        {len(list(model.model.modules()))}")

# Step 2: Load pretrained weights
model.load('yolo11n.pt')

print(f"\n{'='*60}")
print(f"WEIGHTS LOADED")
print(f"{'='*60}")
print(f"Source:        Pretrained weights (transfer learning)")
print(f"Parameters:    {sum(p.numel() for p in model.model.parameters()):,}")
print(f"Trainable:     {sum(p.numel() for p in model.model.parameters() if p.requires_grad):,}")
print(f"{'='*60}")

---
## Cell 4: Training - The Optimized Recipe

All hyperparameters are defined in a single dict and passed to model.train().

### Key changes vs baselines:
- mosaic: 0.0 - Disabled. Mosaic was destroying fine-grained defect features.
- optimizer: AdamW with lr0=0.001 - Better for fine features than SGD.
- mixup: 0.15 - Light regularization.
- copy_paste: 0.1 - Augments with copied defect patches.
- epochs: 600, patience: 150 - More time to converge.

copy_paste requires segmentation masks. If it fails, the notebook
catches the error and retries with copy_paste=0.0.

In [ ]:
# Cell 4: Training - Optimized Recipe
from ultralytics import YOLO
import time
import json as _json

# Build model from YAML + load weights (cell-independent)
model = YOLO('yolo11.yaml')
model.load('yolo11n.pt')

# Training hyperparameters
base_overrides = {
    'data': str(DATA_YAML),
    'task': 'detect',
    'epochs': 600,
    'patience': 150,
    'batch': 16,
    'imgsz': 800,
    'device': 0,
    'optimizer': 'AdamW',
    'lr0': 0.001,
    'lrf': 0.01,
    'momentum': 0.937,
    'weight_decay': 0.0005,
    'warmup_epochs': 5,
    'warmup_momentum': 0.8,
    'warmup_bias_lr': 0.1,
    'mosaic': 0.0,
    'mixup': 0.15,
    'copy_paste': 0.1,
    'copy_paste_mode': 'flip',
    'degrees': 10.0,
    'translate': 0.2,
    'scale': 0.6,
    'shear': 5.0,
    'perspective': 0.0,
    'flipud': 0.5,
    'fliplr': 0.5,
    'hsv_h': 0.015,
    'hsv_s': 0.7,
    'hsv_v': 0.4,
    'erasing': 0.4,
    'cos_lr': True,
    'deterministic': True,
    'close_mosaic': 10,
    'amp': True,
    'seed': 42,
    'workers': 8,
    'save': True,
    'save_period': 50,
    'plots': True,
    'project': str(RUNS_DIR),
    'name': RUN_NAME,
    'exist_ok': True,
    'verbose': True,
}

print(f"{'='*60}")
print("TRAINING CONFIGURATION")
print(f"{'='*60}")
print(f"Architecture:  YOLOv11n (from YAML)")
print(f"Weights:       Pretrained (transfer learning)")
print(f"Dataset:       {DATA_YAML}")
print(f"Epochs:        {base_overrides['epochs']}")
print(f"Patience:      {base_overrides['patience']}")
print(f"Batch:         {base_overrides['batch']}")
print(f"Image size:    {base_overrides['imgsz']}")
print(f"Optimizer:     {base_overrides['optimizer']}")
print(f"Mosaic:        {base_overrides['mosaic']}")
print(f"Mixup:         {base_overrides['mixup']}")
print(f"Copy-paste:    {base_overrides['copy_paste']} ({base_overrides['copy_paste_mode']})")
print(f"Output:        {RUNS_DIR / RUN_NAME}")
print(f"{'='*60}\n")

# Train with copy_paste, fallback to 0.0 if it fails
training_start = time.time()
copy_paste_used = base_overrides['copy_paste']

try:
    print(f"Starting training with copy_paste={copy_paste_used}...")
    results = model.train(**base_overrides)
except Exception as e:
    error_msg = str(e).lower()
    if 'copy_paste' in error_msg or 'segment' in error_msg or 'mask' in error_msg:
        print(f"\ncopy_paste={copy_paste_used} failed (no segmentation masks).")
        print(f"Error: {e}")
        print(f"Retrying with copy_paste removed...\n")
        base_overrides.pop('copy_paste', None)
        copy_paste_used = 0.0
        model = YOLO('yolo11.yaml')
        model.load('yolo11n.pt')
        results = model.train(**base_overrides)
    else:
        raise

training_time = time.time() - training_start

_train_status = {'training_time_seconds': training_time, 'copy_paste_used': copy_paste_used}
_status_file = EVALS_DIR / 'fresh_baseline_train_status.json'
_status_file.parent.mkdir(parents=True, exist_ok=True)
with open(_status_file, 'w') as _f:
    _json.dump(_train_status, _f, indent=2)

print(f"\n{'='*60}")
print(f"TRAINING COMPLETE")
print(f"{'='*60}")
print(f"Duration: {training_time / 3600:.1f} hours")
print(f"Copy-paste: {copy_paste_used}")
print(f"Weights: {BEST_PT}")
print(f"{'='*60}")

---
## Cell 5: Load Best Weights and Evaluate on Test Set

Load the best checkpoint from training and run validation on the test split.
Report mAP@0.5, mAP@0.5:0.95, and per-class AP.

In [ ]:
# Cell 5: Load Best Weights and Evaluate on Test Set
from ultralytics import YOLO
import json
from pathlib import Path

# Safe fallbacks if training cell was not run in this session
import json as _json
_status_path = EVALS_DIR / 'fresh_baseline_train_status.json'
if _status_path.exists():
    with open(_status_path) as _f:
        _status = _json.load(_f)
    training_time = _status.get('training_time_seconds', 0)
    copy_paste_used = _status.get('copy_paste_used', 0.1)
elif 'training_time' not in dir():
    training_time = 0
    copy_paste_used = 0.1

# Verify best weights exist
if not BEST_PT.exists():
    candidates = list((RUNS_DIR / RUN_NAME).rglob('best.pt'))
    if candidates:
        BEST_PT = candidates[0]
        print(f"Using found weights: {BEST_PT}")
    else:
        raise FileNotFoundError(f"No best.pt found in {RUNS_DIR / RUN_NAME}")

print(f"Loading best weights: {BEST_PT}")
best_model = YOLO(str(BEST_PT))

# Run validation on test set
print(f"\nRunning validation on test set...")
val_results = best_model.val(
    data=str(DATA_YAML),
    split='test',
    imgsz=800,
    batch=16,
    device=0,
    plots=True,
    verbose=True,
)

# Extract metrics
map50 = float(val_results.box.map50)
map50_95 = float(val_results.box.map)
precision = float(val_results.box.mp)
recall = float(val_results.box.mr)

# Per-class AP@0.5
class_names = getattr(val_results, 'names', None) or best_model.names
per_class_ap50 = {}
for cls_id, cls_name in class_names.items():
    if cls_id < len(val_results.box.ap50):
        per_class_ap50[cls_name] = float(val_results.box.ap50[cls_id])
    else:
        per_class_ap50[cls_name] = 0.0

print(f"\n{'='*60}")
print(f"TEST SET EVALUATION RESULTS")
print(f"{'='*60}")
print(f"mAP@0.5:      {map50:.4f} ({map50*100:.1f}%)")
print(f"mAP@0.5:0.95: {map50_95:.4f} ({map50_95*100:.1f}%)")
print(f"Precision:    {precision:.4f}")
print(f"Recall:       {recall:.4f}")
print(f"\nPer-class AP@0.5:")
print(f"{'-'*40}")
for cls_name, ap in per_class_ap50.items():
    print(f"  {cls_name:<20} {ap:.4f} ({ap*100:.1f}%)")
print(f"{'='*60}")

# Save metrics to JSON
metrics = {
    "experiment": "fresh_baseline",
    "model": "yolo11n.pt",
    "map50": map50,
    "map50_95": map50_95,
    "precision": precision,
    "recall": recall,
    "per_class_ap50": per_class_ap50,
    "training_time_hours": round(training_time / 3600, 2),
    "copy_paste_used": copy_paste_used,
    "hyperparameters": {
        "epochs": 600,
        "patience": 150,
        "batch": 16,
        "imgsz": 800,
        "optimizer": "AdamW",
        "lr0": 0.001,
        "mosaic": 0.0,
        "mixup": 0.15,
        "copy_paste": copy_paste_used,
        "cos_lr": True,
    },
}

METRICS_JSON.parent.mkdir(parents=True, exist_ok=True)
with open(METRICS_JSON, 'w') as f:
    json.dump(metrics, f, indent=2)
print(f"\nMetrics saved to: {METRICS_JSON}")

---
## Cell 6: Results Visualization

Plot training curves (loss, mAP) from results.csv and per-class AP bar chart.

In [ ]:
# Cell 6: Results Visualization
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
from pathlib import Path

# Load results CSV
if not RESULTS_CSV.exists():
    alt = RUNS_DIR / RUN_NAME / 'results.csv'
    if alt.exists():
        RESULTS_CSV = alt
    else:
        raise FileNotFoundError(f"results.csv not found at {RESULTS_CSV}")

df = pd.read_csv(RESULTS_CSV)
df.columns = df.columns.str.strip()
print(f"Loaded results: {len(df)} epochs")

# Figure 1: Training curves
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle('Fresh Baseline - Training Curves', fontsize=14, fontweight='bold')

loss_cols = [c for c in df.columns if 'loss' in c.lower()]
if loss_cols:
    ax = axes[0, 0]
    for col in loss_cols:
        ax.plot(df[col], label=col, alpha=0.8)
    ax.set_xlabel('Epoch')
    ax.set_ylabel('Loss')
    ax.set_title('Training and Validation Loss')
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)

map50_col = [c for c in df.columns if 'map50' in c.lower() and 'map50-' not in c.lower()]
map_col = [c for c in df.columns if 'map50-95' in c.lower()]
ax = axes[0, 1]
if map50_col:
    ax.plot(df[map50_col[0]], label='mAP@0.5', color='blue', linewidth=2)
if map_col:
    ax.plot(df[map_col[0]], label='mAP@0.5:0.95', color='red', linewidth=2)
ax.set_xlabel('Epoch')
ax.set_ylabel('mAP')
ax.set_title('mAP Progress')
ax.legend()
ax.grid(True, alpha=0.3)

lr_col = [c for c in df.columns if 'lr' in c.lower() or 'pg0' in c.lower()]
ax = axes[1, 0]
if lr_col:
    ax.plot(df[lr_col[0]], color='green', linewidth=2)
    ax.set_xlabel('Epoch')
    ax.set_ylabel('Learning Rate')
    ax.set_title('Learning Rate Schedule')
    ax.grid(True, alpha=0.3)

prec_col = [c for c in df.columns if 'precision' in c.lower()]
rec_col = [c for c in df.columns if 'recall' in c.lower()]
ax = axes[1, 1]
if prec_col:
    ax.plot(df[prec_col[0]], label='Precision', color='purple', linewidth=2)
if rec_col:
    ax.plot(df[rec_col[0]], label='Recall', color='orange', linewidth=2)
ax.set_xlabel('Epoch')
ax.set_ylabel('Score')
ax.set_title('Precision and Recall')
ax.legend()
ax.grid(True, alpha=0.3)

plt.tight_layout()
curves_path = EVALS_DIR / 'fresh_baseline_training_curves.png'
plt.savefig(curves_path, dpi=150, bbox_inches='tight')
plt.show()
print(f"Training curves saved to: {curves_path}")

# Figure 2: Per-class AP bar chart
if 'per_class_ap50' in globals() and per_class_ap50:
    fig2, ax2 = plt.subplots(figsize=(10, 5))
    classes = list(per_class_ap50.keys())
    aps = [v * 100 for v in per_class_ap50.values()]
    colors = plt.cm.Set2(np.linspace(0, 1, len(classes)))
    bars = ax2.bar(classes, aps, color=colors, edgecolor='black', linewidth=0.5)
    ax2.set_ylabel('AP@0.5 (%)')
    ax2.set_title('Fresh Baseline - Per-class AP@0.5', fontweight='bold')
    ax2.set_ylim(0, 100)
    ax2.axhline(y=map50*100, color='red', linestyle='--', linewidth=1.5, label=f'mAP@0.5 = {map50*100:.1f}%')
    for bar, ap in zip(bars, aps):
        ax2.text(bar.get_x() + bar.get_width()/2., bar.get_height() + 1,
                 f'{ap:.1f}%', ha='center', va='bottom', fontsize=10, fontweight='bold')
    ax2.legend()
    ax2.grid(axis='y', alpha=0.3)
    plt.xticks(rotation=15, ha='right')
    plt.tight_layout()
    bars_path = EVALS_DIR / 'fresh_baseline_per_class_ap.png'
    plt.savefig(bars_path, dpi=150, bbox_inches='tight')
    plt.show()
    print(f"Per-class AP chart saved to: {bars_path}")

---
## Cell 7: Comparison Summary - 4A vs Baseline

Compare Fresh Baseline results against the previous best baseline (75.8% mAP@0.5).

In [ ]:
# Cell 7: Comparison Summary
import json
from pathlib import Path

BASELINE_MAP50 = 0.758
delta_map50 = map50 - BASELINE_MAP50

print(f"{'='*65}")
print(f"EXPERIMENT 4A - COMPARISON SUMMARY")
print(f"{'='*65}")
print(f"")
print(f"{'Metric':<25} {'Baseline':>12} {'4A':>12} {'Delta':>12}")
print(f"{'-'*65}")
print(f"{'mAP@0.5':<25} {BASELINE_MAP50*100:>11.1f}% {map50*100:>11.1f}% {delta_map50*100:>+11.1f}%")
print(f"{'mAP@0.5:0.95':<25} {'--':>12} {map50_95*100:>11.1f}% {'':>12}")
print(f"{'Precision':<25} {'--':>12} {precision*100:>11.1f}% {'':>12}")
print(f"{'Recall':<25} {'--':>12} {recall*100:>11.1f}% {'':>12}")

print(f"")
print(f"Per-class AP@0.5 (4A):")
print(f"{'-'*45}")
print(f"{'Class':<22} {'AP@0.5':>12}")
print(f"{'-'*45}")
for cls_name in sorted(per_class_ap50.keys()):
    print(f"{cls_name:<22} {per_class_ap50[cls_name]*100:>11.1f}%")

print(f"")
if delta_map50 > 0:
    print(f"Improvement over baseline: +{delta_map50*100:.1f}%")
elif delta_map50 < 0:
    print(f"Regression from baseline: {delta_map50*100:.1f}%")
else:
    print(f"No change from baseline")

# Save summary
summary = {
    "experiment": "fresh_baseline",
    "baseline_map50": BASELINE_MAP50,
    "current_map50": map50,
    "delta_map50": delta_map50,
    "map50_95": map50_95,
    "precision": precision,
    "recall": recall,
    "per_class_ap50": per_class_ap50,
    "training_time_hours": round(training_time / 3600, 2),
    "copy_paste_used": copy_paste_used,
}

summary_path = EVALS_DIR / 'week4_4a_summary.json'
with open(summary_path, 'w') as f:
    json.dump(summary, f, indent=2)
print(f"\nSummary saved to: {summary_path}")